In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import show


def save_overlay_with_numbers(img_path, geojson_path, output_path):
    """
    Overlay crown polygons on orthomosaic
    and add crown numbering.
    """

    # ---------------------------------------------------
    # Read crown GEOJSON
    # ---------------------------------------------------

    crowns = gpd.read_file(geojson_path)

    # Remove bad geometries
    crowns = crowns[crowns.geometry.notnull()]
    crowns = crowns[crowns.is_valid]
    crowns = crowns[~crowns.geometry.is_empty]

    print(f"Valid crowns: {len(crowns)}")

    # ---------------------------------------------------
    # Open raster and CRS check
    # ---------------------------------------------------

    with rasterio.open(img_path) as src:
        raster_crs = src.crs

    # Reproject crowns if CRS differs
    if crowns.crs != raster_crs:
        crowns = crowns.to_crs(raster_crs)

    # ---------------------------------------------------
    # Create figure
    # ---------------------------------------------------

    fig, ax = plt.subplots(figsize=(14, 14))

    # ---------------------------------------------------
    # Display orthomosaic
    # ---------------------------------------------------

    with rasterio.open(img_path) as src:

        # RGB image
        if src.count >= 3:

            image = src.read([1, 2, 3]).astype(float)

            # Contrast stretching
            for band_index in range(3):

                band = image[band_index]

                valid = band[np.isfinite(band) & (band > 0)]

                if valid.size:

                    p2, p98 = np.percentile(valid, (2, 98))

                    if p98 > p2:

                        image[band_index] = np.clip(
                            (band - p2) / (p98 - p2),
                            0,
                            1
                        )

            show(image, transform=src.transform, ax=ax)

        else:
            show(src, ax=ax, cmap="gray")

    # ---------------------------------------------------
    # Plot crown polygons
    # ---------------------------------------------------

    crowns.plot(
        ax=ax,
        facecolor="white",
        edgecolor="red",
        linewidth=1
    )

    # ---------------------------------------------------
    # Crown numbering
    # ---------------------------------------------------

    for idx, row in crowns.iterrows():

        try:

            geom = row.geometry

            # Skip invalid geometry
            if geom is None or geom.is_empty:
                continue

            centroid = geom.centroid

            ax.text(
                float(centroid.x),
                float(centroid.y),
                str(idx + 1),
                color="black",
                fontsize=8,
                fontweight="bold",
                ha="center",
                va="center"
            )

        except Exception as e:

            print(f"Skipping crown {idx}: {e}")
            continue

    # ---------------------------------------------------
    # Final formatting
    # ---------------------------------------------------

    ax.set_title("Tree Crown Overlay with Numbering")
    ax.set_axis_off()

    plt.tight_layout()

    # ---------------------------------------------------
    # Save output
    # ---------------------------------------------------

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"\nSaved overlay image:")
    print(output_path)



# ---------------------------------------------------
# Example usage
# ---------------------------------------------------

img_path = r"D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\MITTAL\mittal_om1.tif"

crowns_path = r"D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\MITTAL\mittal_om1_crowns.gpkg"

output_path = r"D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\MITTAL\mittal_number_crowns.png"

save_overlay_with_numbers(
    img_path,
    crowns_path,
    output_path
)


Valid crowns: 36

Saved overlay image:
D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\MITTAL\mittal_number_crowns.png


In [3]:
# import geopandas as gpd
# import matplotlib.pyplot as plt
# import numpy as np
# import rasterio
# from rasterio.plot import show


# def save_overlay_with_numbers(img_path, geojson_path, output_path):
#     """
#     Overlay crown polygons on orthomosaic
#     and add crown numbering.
#     """

#     # Read GEOJSON instead of GPKG
#     crowns = gpd.read_file(geojson_path)

#     # Open raster
#     with rasterio.open(img_path) as src:
#         raster_crs = src.crs

#     # Reproject crowns if needed
#     if crowns.crs != raster_crs:
#         crowns = crowns.to_crs(raster_crs)

#     # Create plot
#     fig, ax = plt.subplots(figsize=(14, 14))

#     with rasterio.open(img_path) as src:

#         # RGB display
#         if src.count >= 3:
#             image = src.read([1, 2, 3]).astype(float)

#             # Contrast stretching
#             for band_index in range(3):
#                 band = image[band_index]

#                 valid = band[np.isfinite(band) & (band > 0)]

#                 if valid.size:
#                     p2, p98 = np.percentile(valid, (2, 98))

#                     if p98 > p2:
#                         image[band_index] = np.clip(
#                             (band - p2) / (p98 - p2),
#                             0,
#                             1
#                         )

#             show(image, transform=src.transform, ax=ax)

#         else:
#             show(src, ax=ax, cmap="gray")

#     # Draw crown polygons
#     crowns.plot(
#         ax=ax,
#         facecolor="white",
#         edgecolor="red",
#         linewidth=1
#     )

#     # Crown numbering
#     for idx, row in crowns.iterrows():

#         centroid = row.geometry.centroid

#         ax.text(
#             centroid.x,
#             centroid.y,
#             str(idx + 1),
#             color="black",
#             fontsize=8,
#             fontweight="bold",
#             ha="center",
#             va="center"
#         )

#     ax.set_title("Tree Crown Overlay with Numbering")
#     ax.set_axis_off()

#     plt.tight_layout()

#     fig.savefig(
#         output_path,
#         dpi=300,
#         bbox_inches="tight"
#     )

#     plt.close(fig)

#     print(f"Saved overlay: {output_path}")


# # ---------------------------------------------------
# # Example usage
# # ---------------------------------------------------

# img_path = r"D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\SIT\sit_om1.tif"

# geojson_path = r"D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\consensus_crowns_om1_phenology_sit.geojson"

# output_path = r"D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\sit_check_overlay_numbered.png"

# save_overlay_with_numbers(
#     img_path,
#     geojson_path,
#     output_path
# )